# P300-ViT Leak-Free — Standalone Evaluation (v8 Companion)

Runs the **xDAWN → CWT → ViT** pipeline under the frozen leak-free protocol
(`docs/PROTOCOL.md` in the embedded code), plus the leaky trial-randomized
contrast, the selective-classification (reject option) analysis, and the FAM
stimulus-locking check. **Self-contained**: the code ships inside this
notebook; only the data attachments below are needed.

**Stages:** participant (subject-wise 5-fold CV: clinical / ASZED / pooled)
→ trialwise (leaky contrast) → selective (post-hoc) → fam → consolidated
`xdawn_vit_summary.json`.

| dataset slug | what it is |
|---|---|
| `sabeti-p300` / `p300-ieee` | clinical cohort (private) |
| `aszedeeg` | public ASZED-153 (CC-BY-4.0) |

Enable GPU (T4), Run All. Artifacts land in `/kaggle/working/results/xdawn_vit/`
(+ `xdawn_vit_outputs.zip`).


In [ ]:
!pip install -q pyriemann
# torch / sklearn / scipy / matplotlib / mne are preinstalled on the Kaggle image

In [ ]:
import os, sys, glob

# ---- locate the codebase and the datasets (mount-layout agnostic) ---------- #
def all_input_dirs():
    dirs = []
    for pat in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        dirs += [p for p in glob.glob(pat) if os.path.isdir(p)]
    return dirs

def find_dir(fragment):
    for d in all_input_dirs():
        if fragment in os.path.basename(d):
            return d
    return None

EMBEDDED = "/kaggle/working/embedded_code"
CODE_ROOT = EMBEDDED if os.path.isdir(os.path.join(EMBEDDED, "src")) else EMBEDDED
LOCAL_ROOT = find_dir("p300-ieee") or find_dir("sabeti")
ASZED_ROOT = find_dir("aszedeeg") or find_dir("aszed")
print("CODE_ROOT ", CODE_ROOT)
print("LOCAL_ROOT", LOCAL_ROOT)
print("ASZED_ROOT", ASZED_ROOT)
sys.path.insert(0, os.path.join(CODE_ROOT, "src"))
os.chdir(CODE_ROOT)
assert os.path.isfile("scripts/run_all.py"), "driver missing from code root"


In [ ]:
# ---- run the frozen protocol end-to-end (resumable per stage/experiment) ---- #
!python scripts/run_all.py --stage all --out /kaggle/working/results/xdawn_vit \
    --cache-dir /kaggle/working/cache \
    --local-root "{LOCAL_ROOT}" --aszed-root "{ASZED_ROOT}"

In [ ]:
# ---- consolidated summary + output archive ---------------------------------- #
import json, shutil

SUMMARY = "/kaggle/working/results/xdawn_vit/xdawn_vit_summary.json"
with open(SUMMARY) as f:
    print(json.dumps(json.load(f), indent=2))
shutil.make_archive("/kaggle/working/xdawn_vit_outputs", "zip", "/kaggle/working/results")
print("[done] archive -> /kaggle/working/xdawn_vit_outputs.zip")
